Import

In [29]:
import pandas as pd
import pickle
from sklearn.metrics.pairwise import cosine_similarity

Load Dataset

In [30]:
df = pd.read_csv("../data/processed/merged_dataset.csv")

df.head()

,name,category,rating,address,subtypes,type
0,Taman Budaya Yogyakarta,pusat kebudayaan,4.6,"Jl. Sriwedani No.1, Ngupasan, Kec. Gondomanan,...","Cultural center, Concert hall, Event venue",wisata
1,Vredeburg Fort Museum,museums,4.7,"Jl. Margo Mulyo No.6, Ngupasan, Kec. Gondomana...","Tourist attraction, Fortress, Natural history ...",wisata
2,Situs Pesanggrahan Rejawinangun,attractions,4.4,"Jl. Veteran No.77, Warungboto, Kec. Umbulharjo...","Tourist attraction, Historical landmark",wisata
3,Taman Sari Tourist Village,attractions,4.6,"Patehan, Kraton, Yogyakarta City, Special Regi...","Tourist attraction, Historical landmark",wisata
4,Bentara Budaya Yogyakarta (BBY),attractions,4.6,"Jl. Suroto No.2, Kotabaru, Kec. Gondokusuman, ...","Tourist attraction, Art center",wisata


Load TF-IDF MODEL

In [31]:
with open("../data/processed/tfidf_vectorizer.pkl", "rb") as f:
    vectorizer = pickle.load(f)

with open("../data/processed/tfidf_matrix.pkl", "rb") as f:
    tfidf_matrix = pickle.load(f)

print("Model loaded ✅")

Model loaded ✅


CEK DIMENSI

In [32]:
print("Data:", df.shape)
print("TF-IDF:", tfidf_matrix.shape)

Data: (730, 6)
TF-IDF: (730, 704)


FUNCTION RECOMMEND

In [33]:
def recommend_places(query, df, vectorizer, tfidf_matrix, top_n=5, filter_type=None, min_score=0.1):
    """
    Recommend places based on query similarity with quality filtering.
    
    Args:
        query: User search query (will be cleaned to match training data)
        df: DataFrame with place data
        vectorizer: Fitted TfidfVectorizer
        tfidf_matrix: Pre-computed TF-IDF matrix
        top_n: Number of results to return
        filter_type: Optional filter by 'wisata', 'kuliner', or 'hotel'
        min_score: Minimum similarity score threshold (filters out irrelevant results)
    """
    
    import re
    from sklearn.metrics.pairwise import cosine_similarity
    
    def clean_text(text):
        text = str(text).lower()
        text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text
    
    # CRITICAL: Preprocess query same way as training data
    query_clean = clean_text(query)
    query_vec = vectorizer.transform([query_clean])
    
    # Get similarity scores
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    # Build DataFrame with results and scores
    results = df.copy()
    results['similarity_score'] = sim_scores
    
    # CRITICAL: Filter by type FIRST if specified
    if filter_type:
        results = results[results['type'] == filter_type]
    
    # CRITICAL: Remove zero/low scores (irrelevant results)
    # Only keep results with meaningful similarity
    results = results[results['similarity_score'] > min_score]
    
    # Sort by similarity (highest first)
    results = results.sort_values('similarity_score', ascending=False)
    
    # Return only requested columns and top N results
    return results[["name", "rating", "type", "similarity_score"]].head(top_n)


TEST 1

In [34]:
recommend_places("pantai", df, vectorizer, tfidf_matrix, top_n=5, filter_type="wisata", min_score=0.1)


,name,rating,type,similarity_score
22,Pantai Drini,4.6,wisata,0.530939
23,Pantai Nguyahan,4.6,wisata,0.530939
24,Pantai baru,4.6,wisata,0.468941


TEST 2

In [35]:
recommend_places("cafe kopi", df, vectorizer, tfidf_matrix, top_n=5, filter_type="kuliner", min_score=0.1)


,name,rating,type,similarity_score
341,Kopi Joos Pak Jon (Kopi Arang),4.7,kuliner,0.624495
322,Kedai kopi NATADAMAR,4.7,kuliner,0.592126
328,Warna Kopi,4.7,kuliner,0.587192
276,KOPI PAKPOS NOL KM,4.6,kuliner,0.587192
284,CAFE MACAN OM GABS,5.0,kuliner,0.509155


TEST 3

In [36]:
recommend_places("hotel murah budget", df, vectorizer, tfidf_matrix, top_n=5, filter_type="hotel", min_score=0.1)


,name,rating,type,similarity_score
564,Front One Budget Malioboro Jogja,4.5,hotel,0.437112
710,Ndalem Kampung Budget Stay,4.8,hotel,0.326110
713,Faire BNB Homestay | Penginapan Murah Jogja | ...,4.9,hotel,0.318664
723,PULANG ke UTTARA,4.7,hotel,0.222355
606,Hotel Pakuning,3.8,hotel,0.194287


TEST 4 REAL 

In [37]:
recommend_places("wisata alam", df, vectorizer, tfidf_matrix, top_n=5, min_score=0.1)


,name,rating,type,similarity_score
127,Wisata Alam Jurug Gedhe꧋ꦮꦠꦸꦥꦪꦸꦁ,4.1,wisata,0.823132
667,Alam Jogja Resort,4.3,hotel,0.418623
81,Wisata Kalibiru,4.4,wisata,0.217258
187,RENCANA WISATA DESA BANGUNTAPAN,NaN,wisata,0.181224
199,Desa Wisata Sendari,4.3,wisata,0.157670


In [38]:
# COMPREHENSIVE QUALITY TESTS
print("=" * 70)
print("SYSTEM QUALITY VERIFICATION")
print("=" * 70)
print()

# TEST 1: Beach Query - Should ONLY return wisata (beaches, etc)
print("TEST 1: Query 'pantai' + filter_type='wisata'")
print("-" * 70)
result = recommend_places("Pantai", df, vectorizer, tfidf_matrix, top_n=5, filter_type="wisata", min_score=0.1)
print(f"Results: {len(result)} items")
if len(result) > 0:
    print(f"Types in result: {result['type'].unique()}")
    print(f"Similarity scores: {result['similarity_score'].values}")
    print(result[['name', 'type', 'similarity_score']])
else:
    print("⚠️  No results found (may need lower threshold)")
print()

# TEST 2: Cafe Query - Should return kuliner with cafe keywords
print("TEST 2: Query 'cafe' + filter_type='kuliner'")
print("-" * 70)
result = recommend_places("cafe kopi", df, vectorizer, tfidf_matrix, top_n=5, filter_type="kuliner", min_score=0.1)
print(f"Results: {len(result)} items")
if len(result) > 0:
    print(f"Types in result: {result['type'].unique()}")
    print(f"Similarity scores: {result['similarity_score'].values}")
    print(result[['name', 'type', 'similarity_score']])
else:
    print("⚠️  No results found (may need lower threshold)")
print()

# TEST 3: Hotel Query - Should return ONLY hotel type
print("TEST 3: Query 'hotel murah' + filter_type='hotel'")
print("-" * 70)
result = recommend_places("hotel murah budget", df, vectorizer, tfidf_matrix, top_n=5, filter_type="hotel", min_score=0.1)
print(f"Results: {len(result)} items")
if len(result) > 0:
    print(f"Types in result: {result['type'].unique()}")
    print(f"Similarity scores: {result['similarity_score'].values}")
    print(result[['name', 'type', 'similarity_score']])
else:
    print("⚠️  No results found (may need lower threshold)")
print()

# TEST 4: No type filter - should return mixed but relevant
print("TEST 4: Query 'wisata alam' (no type filter)")
print("-" * 70)
result = recommend_places("wisata alam", df, vectorizer, tfidf_matrix, top_n=5, min_score=0.1)
print(f"Results: {len(result)} items")
if len(result) > 0:
    print(f"Types in result: {result['type'].unique()}")
    print(f"Similarity scores: {result['similarity_score'].values}")
    print(result[['name', 'type', 'similarity_score']])
else:
    print("⚠️  No results found (may need lower threshold)")
print()

# TEST 5: Verify NO ZERO SCORES are returned
print("TEST 5: Verify no irrelevant results (score > 0.1)")
print("-" * 70)
result = recommend_places("pantai", df, vectorizer, tfidf_matrix, top_n=10, min_score=0.1)
zero_count = (result['similarity_score'] == 0).sum()
low_count = (result['similarity_score'] <= 0.1).sum()
print(f"✓ Total results: {len(result)}")
print(f"✓ Zero scores: {zero_count}")
print(f"✓ Low scores (≤0.1): {low_count}")
print(f"✓ Minimum score: {result['similarity_score'].min():.4f}")
print(f"✓ All results > 0.1? {(result['similarity_score'] > 0.1).all()}")


SYSTEM QUALITY VERIFICATION

TEST 1: Query 'pantai' + filter_type='wisata'
----------------------------------------------------------------------
Results: 3 items
Types in result: <StringArray>
['wisata']
Length: 1, dtype: str
Similarity scores: [0.53093867 0.53093867 0.46894076]
               name    type  similarity_score
22     Pantai Drini  wisata          0.530939
23  Pantai Nguyahan  wisata          0.530939
24      Pantai baru  wisata          0.468941

TEST 2: Query 'cafe' + filter_type='kuliner'
----------------------------------------------------------------------
Results: 5 items
Types in result: <StringArray>
['kuliner']
Length: 1, dtype: str
Similarity scores: [0.62449531 0.59212574 0.58719178 0.58719178 0.50915511]
                               name     type  similarity_score
341  Kopi Joos Pak Jon (Kopi Arang)  kuliner          0.624495
322            Kedai kopi NATADAMAR  kuliner          0.592126
328                      Warna Kopi  kuliner          0.587192
276     